# Credit Default Dataset Audit

This notebook is used to inspect the raw credit-default dataset before building the teaching case.

The purpose is not to train a model yet.

We first want to understand:

- the structure of the dataset,
- the prediction target,
- missing or duplicated observations,
- the values stored in categorical variables,
- the repayment-status codes,
- numerical ranges and unusual values,
- and how each feature should eventually be treated during preprocessing.

In [18]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../Data/raw/credit_card_default_raw.csv")

data = pd.read_csv(DATA_PATH)

print("Dataset shape:", data.shape)
display(data.head())

data.info()
print("Columns:")
print(data.columns.tolist())

print("Number of rows:", data.shape[0])
print("Number of columns:", data.shape[1])

Dataset shape: (30000, 25)


,ID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   ID      30000 non-null  int64
 1   X1      30000 non-null  int64
 2   X2      30000 non-null  int64
 3   X3      30000 non-null  int64
 4   X4      30000 non-null  int64
 5   X5      30000 non-null  int64
 6   X6      30000 non-null  int64
 7   X7      30000 non-null  int64
 8   X8      30000 non-null  int64
 9   X9      30000 non-null  int64
 10  X10     30000 non-null  int64
 11  X11     30000 non-null  int64
 12  X12     30000 non-null  int64
 13  X13     30000 non-null  int64
 14  X14     30000 non-null  int64
 15  X15     30000 non-null  int64
 16  X16     30000 non-null  int64
 17  X17     30000 non-null  int64
 18  X18     30000 non-null  int64
 19  X19     30000 non-null  int64
 20  X20     30000 non-null  int64
 21  X21     30000 non-null  int64
 22  X22     30000 non-null  int64
 23  X23     30000 non-null

In [19]:
target_counts = data["Y"].value_counts().sort_index()

target_percentages = (
    data["Y"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

display(target_summary)

missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percentage": data.isna().mean() * 100
})

display(
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
)

duplicate_count = data.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print(
    "Duplicate percentage:",
    round(duplicate_count / len(data) * 100, 2),
    "%"
)

,count,percentage
Y,,
0,23364,77.88
1,6636,22.12


,missing_count,missing_percentage


Duplicate rows: 0
Duplicate percentage: 0.0 %


In [20]:
print("Unique target values:")
print(sorted(data["Y"].unique()))

categorical_columns = ["X2", "X3", "X4"]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(data[column].value_counts().sort_index())



Unique target values:
[np.int64(0), np.int64(1)]

--- X2 ---
X2
1    11888
2    18112
Name: count, dtype: int64

--- X3 ---
X3
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

--- X4 ---
X4
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64


In [21]:
repayment_status_columns = [
    "X6",
    "X7",
    "X8",
    "X9",
    "X10",
    "X11",
]

for column in repayment_status_columns:
    print(f"\n--- {column} ---")
    print(data[column].value_counts().sort_index())


--- X6 ---
X6
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64

--- X7 ---
X7
-2     3782
-1     6050
 0    15730
 1       28
 2     3927
 3      326
 4       99
 5       25
 6       12
 7       20
 8        1
Name: count, dtype: int64

--- X8 ---
X8
-2     4085
-1     5938
 0    15764
 1        4
 2     3819
 3      240
 4       76
 5       21
 6       23
 7       27
 8        3
Name: count, dtype: int64

--- X9 ---
X9
-2     4348
-1     5687
 0    16455
 1        2
 2     3159
 3      180
 4       69
 5       35
 6        5
 7       58
 8        2
Name: count, dtype: int64

--- X10 ---
X10
-2     4546
-1     5539
 0    16947
 2     2626
 3      178
 4       84
 5       17
 6        4
 7       58
 8        1
Name: count, dtype: int64

--- X11 ---
X11
-2     4895
-1     5740
 0    16286
 2     2766
 3      184
 4       49
 5       13
 6       19
 7       46
 8        2
Name: co

In [22]:
display(data.describe().T)

range_summary = pd.DataFrame({
    "minimum": data.min(),
    "maximum": data.max(),
    "unique_values": data.nunique()
})

display(range_summary)

,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
X1,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
X2,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
X3,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
X4,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
X5,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
X6,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
X7,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
X8,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
X9,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


,minimum,maximum,unique_values
ID,1,30000,30000
X1,10000,1000000,81
X2,1,2,2
X3,0,6,7
X4,0,3,4
X5,21,79,56
X6,-2,8,11
X7,-2,8,11
X8,-2,8,11
X9,-2,8,11


In [23]:
feature_dictionary = pd.DataFrame({
    "original_name": [
        "ID",
        "X1", "X2", "X3", "X4", "X5",
        "X6", "X7", "X8", "X9", "X10", "X11",
        "X12", "X13", "X14", "X15", "X16", "X17",
        "X18", "X19", "X20", "X21", "X22", "X23",
        "Y"
    ],

    "friendly_name": [
        "customer_id",
        "credit_limit",
        "sex",
        "education",
        "marital_status",
        "age",
        "payment_status_sep",
        "payment_status_aug",
        "payment_status_jul",
        "payment_status_jun",
        "payment_status_may",
        "payment_status_apr",
        "bill_amount_sep",
        "bill_amount_aug",
        "bill_amount_jul",
        "bill_amount_jun",
        "bill_amount_may",
        "bill_amount_apr",
        "payment_amount_sep",
        "payment_amount_aug",
        "payment_amount_jul",
        "payment_amount_jun",
        "payment_amount_may",
        "payment_amount_apr",
        "default_next_month"
    ]
})

display(feature_dictionary)

,original_name,friendly_name
0,ID,customer_id
1,X1,credit_limit
2,X2,sex
3,X3,education
4,X4,marital_status
5,X5,age
6,X6,payment_status_sep
7,X7,payment_status_aug
8,X8,payment_status_jul
9,X9,payment_status_jun


In [24]:
feature_dictionary["feature_type"] = [
    "identifier",     # ID
    "numeric",        # X1
    "categorical",    # X2
    "categorical",    # X3
    "categorical",    # X4
    "numeric",        # X5
    "categorical",    # X6
    "categorical",    # X7
    "categorical",    # X8
    "categorical",    # X9
    "categorical",    # X10
    "categorical",    # X11
    "numeric",        # X12
    "numeric",        # X13
    "numeric",        # X14
    "numeric",        # X15
    "numeric",        # X16
    "numeric",        # X17
    "numeric",        # X18
    "numeric",        # X19
    "numeric",        # X20
    "numeric",        # X21
    "numeric",        # X22
    "numeric",        # X23
    "target"          # Y
]

display(feature_dictionary)

,original_name,friendly_name,feature_type
0,ID,customer_id,identifier
1,X1,credit_limit,numeric
2,X2,sex,categorical
3,X3,education,categorical
4,X4,marital_status,categorical
5,X5,age,numeric
6,X6,payment_status_sep,categorical
7,X7,payment_status_aug,categorical
8,X8,payment_status_jul,categorical
9,X9,payment_status_jun,categorical


In [25]:
feature_dictionary["planned_treatment"] = [
    "drop from model",            # ID
    "scale",                      # X1
    "one-hot encode",             # X2
    "clean + one-hot encode",     # X3
    "clean + one-hot encode",     # X4
    "scale",                      # X5
    "one-hot encode",             # X6
    "one-hot encode",             # X7
    "one-hot encode",             # X8
    "one-hot encode",             # X9
    "one-hot encode",             # X10
    "one-hot encode",             # X11
    "scale",                      # X12
    "scale",                      # X13
    "scale",                      # X14
    "scale",                      # X15
    "scale",                      # X16
    "scale",                      # X17
    "scale",                      # X18
    "scale",                      # X19
    "scale",                      # X20
    "scale",                      # X21
    "scale",                      # X22
    "scale",                      # X23
    "prediction target"           # Y
]

display(feature_dictionary)

,original_name,friendly_name,feature_type,planned_treatment
0,ID,customer_id,identifier,drop from model
1,X1,credit_limit,numeric,scale
2,X2,sex,categorical,one-hot encode
3,X3,education,categorical,clean + one-hot encode
4,X4,marital_status,categorical,clean + one-hot encode
5,X5,age,numeric,scale
6,X6,payment_status_sep,categorical,one-hot encode
7,X7,payment_status_aug,categorical,one-hot encode
8,X8,payment_status_jul,categorical,one-hot encode
9,X9,payment_status_jun,categorical,one-hot encode


In [26]:
feature_dictionary["notes"] = ""

feature_dictionary.loc[
    feature_dictionary["original_name"] == "X3",
    "notes"
] = "Contains undocumented category codes 0, 5 and 6."

feature_dictionary.loc[
    feature_dictionary["original_name"] == "X4",
    "notes"
] = "Contains undocumented category code 0."

feature_dictionary.loc[
    feature_dictionary["original_name"].isin(
        ["X6", "X7", "X8", "X9", "X10", "X11"]
    ),
    "notes"
] = "Contains repayment-status codes -2 and 0; keep as separate categories initially."

display(feature_dictionary)

,original_name,friendly_name,feature_type,planned_treatment,notes
0,ID,customer_id,identifier,drop from model,
1,X1,credit_limit,numeric,scale,
2,X2,sex,categorical,one-hot encode,
3,X3,education,categorical,clean + one-hot encode,"Contains undocumented category codes 0, 5 and 6."
4,X4,marital_status,categorical,clean + one-hot encode,Contains undocumented category code 0.
5,X5,age,numeric,scale,
6,X6,payment_status_sep,categorical,one-hot encode,Contains repayment-status codes -2 and 0; keep...
7,X7,payment_status_aug,categorical,one-hot encode,Contains repayment-status codes -2 and 0; keep...
8,X8,payment_status_jul,categorical,one-hot encode,Contains repayment-status codes -2 and 0; keep...
9,X9,payment_status_jun,categorical,one-hot encode,Contains repayment-status codes -2 and 0; keep...


In [27]:
from pathlib import Path

METADATA_PATH = Path("../Data/metadata")
METADATA_PATH.mkdir(parents=True, exist_ok=True)

feature_dictionary.to_csv(
    METADATA_PATH / "credit_feature_dictionary.csv",
    index=False
)

print("Feature dictionary saved.")

Feature dictionary saved.
